# Prepare xView Dataset for YOLO Training

This notebook downloads the **pre-processed** xView dataset from MinIO.

The dataset has been converted locally from 8-band multispectral GeoTIFF to RGB PNG format and is ready to use.

**What's included:**
- 1,127 PNG images (converted from 8-band multispectral GeoTIFF)
- 846 YOLO format label files
- Train/val split files
- xView.yaml configuration

**Run this notebook once** to prepare the dataset, then use `train-yolo-xview.ipynb` for training.

## 1. Install Dependencies

In [ ]:
!pip install minio

## 2. Configure MinIO Connection

In [ ]:
import os
from minio import Minio
from pathlib import Path

# MinIO configuration
AWS_S3_ENDPOINT = os.getenv("AWS_S3_ENDPOINT", "minio-api-minio.apps.ocp.z8lpk.sandbox1242.opentlc.com")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "minio123")
AWS_S3_BUCKET = os.getenv("AWS_S3_BUCKET", "demo")

# Strip protocol from endpoint
endpoint = AWS_S3_ENDPOINT.replace('https://', '').replace('http://', '').rstrip('/')

# Create MinIO client
client = Minio(
    endpoint,
    access_key=AWS_ACCESS_KEY_ID,
    secret_key=AWS_SECRET_ACCESS_KEY,
    secure=True
)

print(f"✓ Connected to MinIO: {endpoint}")
print(f"✓ Using bucket: {AWS_S3_BUCKET}")

## 3. Download Pre-Processed Dataset

The dataset has been pre-processed locally:
- Original 8-band multispectral GeoTIFF images converted to RGB PNG
- Labels converted from GeoJSON to YOLO format
- Train/val split files generated

This saves significant time on new cluster provisioning!

In [ ]:
import tarfile

# Download preprocessed dataset tarball
tarball_path = "xView_preprocessed.tar.gz"
dataset_root = Path("./xView")

if dataset_root.exists() and len(list(dataset_root.glob("images/*.png"))) > 0:
    print("✓ Dataset already exists locally, skipping download")
    print(f"  Images: {len(list(dataset_root.glob('images/*.png')))}")
    print(f"  Labels: {len(list(dataset_root.glob('labels/*.txt')))}")
else:
    print("Downloading pre-processed dataset from MinIO...")
    print(f"  Object: xView_preprocessed.tar.gz (~14GB)")
    
    # Download tarball
    client.fget_object(
        AWS_S3_BUCKET,
        "xView_preprocessed.tar.gz",
        tarball_path
    )
    
    print("✓ Downloaded tarball")
    print("\nExtracting dataset...")
    
    # Extract tarball
    with tarfile.open(tarball_path, 'r:gz') as tar:
        tar.extractall(path="xView")
    
    print("✓ Extracted dataset")
    
    # Clean up tarball
    Path(tarball_path).unlink()
    print("✓ Cleaned up tarball")
    
    # Verify extraction
    images = list(dataset_root.glob("images_png/*.png"))
    labels = list(dataset_root.glob("labels/*.txt"))
    
    print(f"\nDataset extracted:")
    print(f"  Images: {len(images)} PNG files")
    print(f"  Labels: {len(labels)} YOLO files")

## 4. Rename images_png to images

The tarball contains `images_png/` but YOLO expects `images/`

In [ ]:
import shutil

images_png = dataset_root / "images_png"
images_dir = dataset_root / "images"

if images_png.exists() and not images_dir.exists():
    images_png.rename(images_dir)
    print("✓ Renamed images_png/ to images/")
elif images_dir.exists():
    print("✓ images/ directory already exists")
else:
    print("⚠ Neither images_png/ nor images/ found")

## 5. Create YOLO Dataset Configuration

In [ ]:
# Create YOLO dataset YAML
dataset_path = Path.cwd() / "xView"
yaml_path = Path.cwd() / "xView.yaml"

yaml_content = f"""path: {dataset_path}
train: images/autosplit_train.txt
val: images/autosplit_val.txt

names:
  0: Fixed-wing Aircraft
  1: Small Aircraft
  2: Cargo Plane
  3: Helicopter
  4: Passenger Vehicle
  5: Small Car
  6: Bus
  7: Pickup Truck
  8: Utility Truck
  9: Truck
  10: Cargo Truck
  11: Truck w/Box
  12: Truck Tractor
  13: Trailer
  14: Truck w/Flatbed
  15: Truck w/Liquid
  16: Crane Truck
  17: Railway Vehicle
  18: Passenger Car
  19: Cargo Car
  20: Flat Car
  21: Tank car
  22: Locomotive
  23: Maritime Vessel
  24: Motorboat
  25: Sailboat
  26: Tugboat
  27: Barge
  28: Fishing Vessel
  29: Ferry
  30: Yacht
  31: Container Ship
  32: Oil Tanker
  33: Engineering Vehicle
  34: Tower crane
  35: Container Crane
  36: Reach Stacker
  37: Straddle Carrier
  38: Mobile Crane
  39: Dump Truck
  40: Haul Truck
  41: Scraper/Tractor
  42: Front loader/Bulldozer
  43: Excavator
  44: Cement Mixer
  45: Ground Grader
  46: Hut/Tent
  47: Shed
  48: Building
  49: Aircraft Hangar
  50: Damaged Building
  51: Facility
  52: Construction Site
  53: Vehicle Lot
  54: Helipad
  55: Storage Tank
  56: Shipping container lot
  57: Shipping Container
  58: Pylon
  59: Tower
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✓ Created {yaml_path}")
print(f"  Dataset path: {dataset_path}")

## 6. Fix Split Files with Absolute Paths

YOLO requires absolute paths in the split files to correctly resolve image locations.

## 7. Verify Dataset Structure

In [ ]:
# Fix train split with absolute paths
train_split = images_dir / "autosplit_train.txt"
with open(train_split) as f:
    train_images = [line.strip() for line in f]

print(f"Updating {len(train_images)} train images to absolute paths...")
with open(train_split, 'w') as f:
    for img_path in train_images:
        # Convert images/1881.png -> /full/path/to/xView/images/1881.png
        abs_path = dataset_path / img_path
        f.write(f"{abs_path}\n")

# Fix val split with absolute paths
val_split = images_dir / "autosplit_val.txt"
with open(val_split) as f:
    val_images = [line.strip() for line in f]

print(f"Updating {len(val_images)} val images to absolute paths...")
with open(val_split, 'w') as f:
    for img_path in val_images:
        abs_path = dataset_path / img_path
        f.write(f"{abs_path}\n")

print("\n✓ Updated split files with absolute paths")

# Verify first entry
print("\nFirst train entry:")
with open(train_split) as f:
    print(f"  {f.readline().strip()}")

In [ ]:
print("=" * 60)
print("Dataset Preparation Complete!")
print("=" * 60)

# Count files
images_dir = dataset_root / "images"
labels_dir = dataset_root / "labels"

images = list(images_dir.glob("*.png"))
labels = list(labels_dir.glob("*.txt"))

# Read split files
with open(images_dir / 'autosplit_train.txt') as f:
    train_count = len(f.readlines())
with open(images_dir / 'autosplit_val.txt') as f:
    val_count = len(f.readlines())

print(f"\nDataset structure:")
print(f"  xView/")
print(f"  ├── images/           {len(images)} PNG files (RGB from 8-band multispectral)")
print(f"  │   ├── autosplit_train.txt ({train_count} images)")
print(f"  │   └── autosplit_val.txt ({val_count} images)")
print(f"  └── labels/           {len(labels)} YOLO label files")
print(f"\nConfiguration: xView.yaml")

# Verify a sample image/label pair
if images:
    sample_image = images[0]
    sample_label = labels_dir / sample_image.with_suffix('.txt').name
    
    print(f"\nSample verification:")
    print(f"  Image: {sample_image.name} exists: ✓")
    print(f"  Label: {sample_label.name} exists: {'✓' if sample_label.exists() else '✗'}")

print(f"\n{'='*60}")
print("Ready for training! Use 'train-yolo-xview.ipynb' to train.")
print(f"{'='*60}")
print(f"\nNote: Images were pre-processed locally from 8-band multispectral")
print(f"      GeoTIFF to RGB PNG format for YOLO compatibility.")